In [2]:
%matplotlib qt
import torch
from d2l import torch as d2l
from torch import nn as nn
import matplotlib.pyplot as plt
import numpy as np

In [3]:
#计算函数
def f(x):
    return x**2
##fig2
#psi_roll as functions of psi_pitch and psi_0
def psi_roll_from_psi_pitch_and_psi_0(psi_pitch,psi_0):
    return psi_0 - torch.arctan(torch.tan(psi_0) * torch.cos(psi_pitch))
    
#psi_yaw as functions of psi_pitch and psi_0
def psi_yaw_from_psi_pitch_and_psi_0(psi_pitch,psi_0):
    return - torch.arcsin(torch.sin(psi_0) * torch.sin(psi_pitch))

def psi_flex_from_psi_pitch_and_psi_0(psi_pitch,psi_0):
    return torch.arcsin((torch.sin(psi_pitch)) / (torch.sqrt(1 + torch.tan(psi_0) ** 2 * torch.cos(psi_pitch) ** 2)))

#fig3
def psi_diff_from_psi_pitch_and_psi_0(psi_pitch,psi_0):

    
    psi_0_abs = torch.abs(psi_0)
    psi_pitch_abs = torch.abs(psi_pitch)
    psi_roll = psi_roll_from_psi_pitch_and_psi_0(psi_pitch_abs,psi_0_abs)
    psi_yaw = psi_yaw_from_psi_pitch_and_psi_0(psi_pitch_abs,psi_0_abs)
    psi_flex = psi_flex_from_psi_pitch_and_psi_0(psi_pitch_abs,psi_0_abs)

    psi_diff = psi_pitch_abs + psi_roll - psi_yaw - psi_flex
    return psi_diff
    

def derivative_psi_0_multi_pitch(psi_0, psi_pitch):
    """
    计算 psi_diff_from_psi_pitch_and_psi_0 关于 psi_0 的导数。
    
    参数:
        psi_0 (torch.Tensor): psi_0 的值（张量）
        psi_pitch (float): 单个 psi_pitch 值（浮点数）
    
    返回:
        torch.Tensor: d(psi_diff)/d(psi_0) 的导数值
    """
    psi_pitch = torch.tensor(psi_pitch, dtype=torch.float32)
    derivs = torch.zeros_like(psi_0)
    
    for i, psi_0_val in enumerate(psi_0):
        if abs(psi_0_val.item()) < 0.0001:  # 跳过 psi_0 ≈ 0
            derivs[i] = float('nan')
            continue
        psi_0_scalar = torch.tensor(psi_0_val.item(), requires_grad=True)
        psi_diff = psi_diff_from_psi_pitch_and_psi_0(psi_pitch, psi_0_scalar)
        psi_diff.backward()
        derivs[i] = psi_0_scalar.grad if psi_0_scalar.grad is not None else float('nan')
    
    return derivs
    
#fig4
def psi_pitch_from_psi_flex_and_psi_0(psi_flex,psi_0):
    return torch.arctan(torch.tan(psi_flex) / torch.cos(psi_0))

def psi_roll_from_psi_flex_and_psi_0(psi_flex,psi_0):
    return psi_0 - torch.arcsin(torch.sin(psi_0) * torch.cos(psi_flex))

def psi_yaw_from_psi_flex_and_psi_0(psi_flex,psi_0):
    return - torch.arctan(torch.tan(psi_0) * torch.sin(psi_flex))

In [14]:
#plt.figure(figsize=(10,3.9),dpi=300)
#plt.rc('text', usetex=True)
#plt.rc('font', family='serif')

#绘制图像fig2(a)
#plt.subplot(1,2,1)

#计算psi_0=pi/4
psi_0 = torch.tensor(torch.pi*38/180, dtype=torch.float32) 
psi_pitch = torch.arange(-torch.pi/2, torch.pi/2+0.1, 0.2)
psi_pitch[-1] = torch.pi/2
psi_roll_a_1 = psi_roll_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
psi_yaw_a_1 = psi_yaw_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
#plt.plot(psi_pitch,psi_roll_a_1,label=r'$\psi_{\mathrm{roll}},\psi_{0}=\frac{\pi}{4}$', linestyle='-',color='#16FF9A',linewidth=1.5)
#plt.plot(psi_pitch,psi_yaw_a_1,label=r'$\psi_{\mathrm{yaw}},\psi_{0}=\frac{\pi}{4}$', linestyle='-',color='#C053FF',linewidth=1.5)
psi_pitch[0],psi_roll_a_1[0],psi_yaw_a_1[0]

(tensor(-1.5708), tensor(0.6632), tensor(0.6632))

In [12]:
plt.figure(figsize=(10,3.9),dpi=300)
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

#绘制图像fig2(a)
plt.subplot(1,2,1)

#计算psi_0=pi/4
psi_0 = torch.tensor(torch.pi/4, dtype=torch.float32) 
psi_pitch = torch.arange(-torch.pi/2, torch.pi/2+0.1, 0.2)
psi_pitch[-1] = torch.pi/2
psi_roll_a_1 = psi_roll_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
psi_yaw_a_1 = psi_yaw_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_pitch,psi_roll_a_1,label=r'$\psi_{\mathrm{roll}},\psi_{0}=\frac{\pi}{4}$', linestyle='-',color='#16FF9A',linewidth=1.5)
plt.plot(psi_pitch,psi_yaw_a_1,label=r'$\psi_{\mathrm{yaw}},\psi_{0}=\frac{\pi}{4}$', linestyle='-',color='#C053FF',linewidth=1.5)
#计算psi_0=-pi/4
psi_0 = torch.tensor(-torch.pi/4, dtype=torch.float32) 
psi_roll_a_2 = psi_roll_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
psi_yaw_a_2 = psi_yaw_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_pitch,psi_roll_a_2,label=r'$\psi_{\mathrm{roll}},\psi_{0}=-\frac{\pi}{4}$', linestyle='--',linewidth=1,color='#16FF9A',marker='o',markersize='4')
plt.plot(psi_pitch,psi_yaw_a_2,label=r'$\psi_{\mathrm{yaw}},\psi_{0}=-\frac{\pi}{4}$', linestyle='--',linewidth=1,color='#C053FF',marker='o',markersize='4')
plt.xlabel(r"$\psi_{\mathrm{pitch}}$",fontsize=15)



plt.grid()
x_ticks = np.arange(-np.pi/2, np.pi/2+0.0001, np.pi/6)
x_tick_labels = [r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{3}$', r'$-\frac{\pi}{6}$',
                 '0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$']
plt.xticks(x_ticks, x_tick_labels,fontsize=20)

# 设置y轴，pi作为基准
y_ticks = np.arange(-np.pi/2, np.pi/2+0.0001, np.pi/6)
y_tick_labels = [r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{3}$', r'$-\frac{\pi}{6}$',
                 '0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$']
plt.yticks(y_ticks, y_tick_labels,fontsize=20)
plt.axis('equal')
plt.legend(prop={'size':15},loc=9,handlelength=1,borderpad=0.2, labelspacing=0.1,bbox_to_anchor=(0.5, 1.05))

plt.xlim(-np.pi/2, np.pi/2)
plt.ylim(-np.pi/2,np.pi/2)
#绘制图像fig2(b)
plt.subplot(1,2,2)

#计算psi_0=0
psi_0 = torch.tensor(0, dtype=torch.float32) 
psi_pitch = torch.arange(-torch.pi/2, torch.pi/2, 0.01)
psi_flex = psi_flex_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_pitch,psi_flex,label=r'$\psi_{\mathrm{hip,flex}},\psi_{0}=0$',linewidth=1.5,linestyle='-')

#计算psi_0=|pi/6|
psi_0 = torch.tensor(torch.pi/6, dtype=torch.float32) 
psi_flex = psi_flex_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_pitch,psi_flex,label=r'$\psi_{\mathrm{hip,flex}},|\psi_{0}|=\frac{\pi}{6}$',linewidth=1.5,linestyle='--')

#计算psi_0=|pi/4|
psi_0 = torch.tensor(torch.pi/4, dtype=torch.float32) 
psi_flex = psi_flex_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_pitch,psi_flex,label=r'$\psi_{\mathrm{hip,flex}},|\psi_{0}|=\frac{\pi}{4}$',linewidth=1.5,linestyle='-.')

#计算psi_0=|pi/4|
psi_0 = torch.tensor(torch.pi/3, dtype=torch.float32) 
psi_flex = psi_flex_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_pitch,psi_flex,label=r'$\psi_{\mathrm{hip,flex}},|\psi_{0}|=\frac{\pi}{3}$',linewidth=1.5,linestyle=':')
.5
plt.grid()

x_ticks = np.arange(-np.pi/2, np.pi/2+0.0001, np.pi/6)
x_tick_labels = [r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{3}$', r'$-\frac{\pi}{6}$',
                 '0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$']
plt.xticks(x_ticks, x_tick_labels,fontsize=20)

# 设置y轴，pi作为基准
y_ticks = np.arange(-np.pi/2, np.pi/2+0.0001, np.pi/6)
y_tick_labels = [r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{3}$', r'$-\frac{\pi}{6}$',
                 '0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$']
plt.yticks(y_ticks, y_tick_labels,fontsize=20)
plt.axis('equal')
plt.legend(prop = {'size':15},loc=2,handlelength=1,borderpad=0.2, labelspacing=0.1,bbox_to_anchor=(0, 1.05))
plt.xlim(-np.pi/2, np.pi/2)
plt.ylim(-np.pi/2,np.pi/2)

(-1.5707963267948966, 1.5707963267948966)

In [2]:
#fig3(a)
plt.figure(figsize=(5.5,3.70),dpi=300)
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

#计算psi_0=0
psi_0 = torch.tensor(0, dtype=torch.float32) 
psi_pitch = torch.arange(-torch.pi/2, torch.pi/2, 0.01)
psi_diff = psi_diff_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_pitch,psi_diff,label=r'$\psi_{0}=0$',linewidth=2,linestyle='-')

#计算psi_0=pi/6
psi_0 = torch.tensor(torch.pi/6, dtype=torch.float32) 
psi_diff = psi_diff_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_pitch,psi_diff,label=r'$|\psi_{0}|=\frac{\pi}{6}$',linewidth=1.5,linestyle='--')

#计算psi_0=pi/4
psi_0 = torch.tensor(torch.pi/4, dtype=torch.float32) 
psi_diff = psi_diff_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_pitch,psi_diff,label=r'$|\psi_{0}|=\frac{\pi}{4}$',linewidth=1.5,linestyle='-.')

#计算psi_0=pi/3
psi_0 = torch.tensor(torch.pi/3, dtype=torch.float32) 
psi_diff = psi_diff_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_pitch,psi_diff,label=r'$|\psi_{0}|=\frac{\pi}{3}$',linewidth=1.5,linestyle=':')

plt.grid()

x_ticks = np.arange(-np.pi/2, np.pi/2+0.0001, np.pi/6)
x_tick_labels = [r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{3}$', r'$-\frac{\pi}{6}$',
                 '0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$']
plt.xticks(x_ticks, x_tick_labels,fontsize=20)

# 设置y轴，pi作为基准
y_ticks = np.arange(0, np.pi*2/3+0.0001, np.pi/6)
y_tick_labels = ['0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$',r'$\frac{2}{3}\pi$']
plt.yticks(y_ticks, y_tick_labels,fontsize=20)
plt.axis('equal')
plt.legend(prop={'size':15},handlelength=1,borderpad=0.2, labelspacing=0.1)
plt.xlim(-np.pi/2,np.pi/2)
plt.ylim(-0.01,np.pi*2/3)
plt.ylabel(r"$\psi_{\mathrm{diff}}$",fontsize=20)

NameError: name 'plt' is not defined

In [4]:
#fig3(b)(c)
plt.figure(figsize=(10,3.74),dpi=300)
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

#绘制图像fig3(b)
plt.subplot(1,2,1)
#计算psi_pitch=0
psi_0 = torch.arange(-torch.pi/2+0.01, torch.pi/2, 0.01)
psi_pitch = torch.tensor(0, dtype=torch.float32) 
psi_diff = psi_diff_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_0,psi_diff,label=r'$\psi_{\mathrm{pitch}}=0$',linewidth=2,linestyle='-')

#计算psi_pitch=pi/6
psi_pitch = torch.tensor(torch.pi/6, dtype=torch.float32) 
psi_diff = psi_diff_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_0,psi_diff,label=r'$|\psi_{\mathrm{pitch}}|=\frac{\pi}{6}$',linewidth=2,linestyle='--')

#计算psi_pitch=pi/4
psi_pitch = torch.tensor(torch.pi/4, dtype=torch.float32) 
psi_diff = psi_diff_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_0,psi_diff,label=r'$|\psi_{\mathrm{pitch}}|=\frac{\pi}{4}$',linewidth=2,linestyle='-.')

#计算psi_pitch=pi/3
psi_pitch = torch.tensor(torch.pi/3, dtype=torch.float32) 
psi_diff = psi_diff_from_psi_pitch_and_psi_0(psi_pitch,psi_0)
plt.plot(psi_0,psi_diff,label=r'$|\psi_{\mathrm{pitch}}|=\frac{\pi}{3}$',linewidth=2,linestyle=':')
plt.grid()

x_ticks = np.arange(-np.pi/2, np.pi/2+0.0001, np.pi/6)
x_tick_labels = [r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{3}$', r'$-\frac{\pi}{6}$',
                 '0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$']
plt.xticks(x_ticks, x_tick_labels,fontsize=20)

y_ticks = np.arange(0, np.pi+0.0001, np.pi/6)
y_tick_labels = ['0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$',r'$\frac{2}{3}\pi$',r'$\frac{5}{6}\pi$',r'$\pi$']
plt.yticks(y_ticks, y_tick_labels,fontsize=20)
plt.axis('equal')
plt.legend(prop={'size':15},handlelength=1,borderpad=0.2, labelspacing=0.1,loc=9)
plt.xlim(-np.pi/2,np.pi/2)
plt.ylim(0,np.pi)
plt.ylabel(r"$\psi_{\mathrm{diff}}$",fontsize=20)
plt.xlabel(r"$\psi_{0}$",fontsize=20)
#绘制图像fig3(c)
plt.subplot(1,2,2)

#psi_pitch = 0
psi_pitch =0
# 定义 psi_0 范围，分段排除 0 点
psi_0_vals_neg = torch.linspace(-torch.pi/2+0.01 , -0.001, 500)  # 负半轴
psi_0_vals_pos = torch.linspace(0.001, torch.pi/2-0.01, 500)     # 正半轴
psi_0_vals = torch.cat([psi_0_vals_neg, psi_0_vals_pos])      # 拼接
# 计算导数
psi_diff_derivs = derivative_psi_0_multi_pitch(psi_0_vals, psi_pitch)
# 分段提取负半轴和正半轴数据
mask_neg = psi_0_vals <= -0.01
mask_pos = psi_0_vals >= 0.01
psi_0_neg = psi_0_vals[mask_neg]
psi_0_pos = psi_0_vals[mask_pos]
derivs_neg = psi_diff_derivs[mask_neg]
derivs_pos = psi_diff_derivs[mask_pos]
plt.plot(psi_0_neg,derivs_neg,label=r"$\psi_{\mathrm{pitch}}=0$",color='C0',linewidth=2,linestyle='-')
plt.plot(psi_0_pos,derivs_pos,color='C0',linewidth=2,linestyle='-')

#psi_pitch = pi/6
psi_pitch =np.pi/6 

# 计算导数
psi_diff_derivs = derivative_psi_0_multi_pitch(psi_0_vals, psi_pitch)
# 分段提取负半轴和正半轴数据
mask_neg = psi_0_vals <= -0.01
mask_pos = psi_0_vals >= 0.01
psi_0_neg = psi_0_vals[mask_neg]
psi_0_pos = psi_0_vals[mask_pos]
derivs_neg = psi_diff_derivs[mask_neg]
derivs_pos = psi_diff_derivs[mask_pos]
plt.plot(psi_0_neg,derivs_neg,label=r"$|\psi_{\mathrm{pitch}}|=\frac{\pi}{6}$",color='C1',linewidth=2,linestyle='--')
plt.plot(psi_0_pos,derivs_pos,color='C1',linewidth=2,linestyle='--')

#psi_pitch = pi/4
psi_pitch =np.pi/4 

# 计算导数
psi_diff_derivs = derivative_psi_0_multi_pitch(psi_0_vals, psi_pitch)
# 分段提取负半轴和正半轴数据
mask_neg = psi_0_vals <= -0.01
mask_pos = psi_0_vals >= 0.01
psi_0_neg = psi_0_vals[mask_neg]
psi_0_pos = psi_0_vals[mask_pos]
derivs_neg = psi_diff_derivs[mask_neg]
derivs_pos = psi_diff_derivs[mask_pos]
plt.plot(psi_0_neg,derivs_neg,label=r"$|\psi_{\mathrm{pitch}}|=\frac{\pi}{4}$",color='C2',linewidth=2,linestyle='-.')
plt.plot(psi_0_pos,derivs_pos,color='C2',linewidth=2,linestyle='-.')

#psi_pitch = pi/3
psi_pitch =np.pi/3 

# 计算导数
psi_diff_derivs = derivative_psi_0_multi_pitch(psi_0_vals, psi_pitch)
# 分段提取负半轴和正半轴数据
mask_neg = psi_0_vals <= -0.01
mask_pos = psi_0_vals >= 0.01
psi_0_neg = psi_0_vals[mask_neg]
psi_0_pos = psi_0_vals[mask_pos]
derivs_neg = psi_diff_derivs[mask_neg]
derivs_pos = psi_diff_derivs[mask_pos]
plt.plot(psi_0_neg,derivs_neg,label=r"$|\psi_{\mathrm{pitch}}|=\frac{\pi}{3}$",color='C3',linewidth=2,linestyle=':')
plt.plot(psi_0_pos,derivs_pos,color='C3',linewidth=2,linestyle=':')

#网格
plt.grid()

x_ticks = np.arange(-np.pi/2, np.pi/2+0.0001, np.pi/6)
x_tick_labels = [r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{3}$', r'$-\frac{\pi}{6}$',
                 '0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$']
plt.xticks(x_ticks, x_tick_labels,fontsize=20)

y_ticks = np.arange(-np.pi/2, np.pi/2+0.0001, np.pi/6)
y_tick_labels = [r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{3}$', r'$-\frac{\pi}{6}$',
                 '0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$']
plt.yticks(y_ticks, y_tick_labels,fontsize=20)
plt.axis('equal')
plt.legend(prop={'size':15},handlelength=1,borderpad=0.2, labelspacing=0.1,bbox_to_anchor=(0.53, 1))
plt.xlim(-np.pi/2,np.pi/2)
plt.ylim(-np.pi/2, np.pi/2)
plt.ylabel(r"${d \psi _{\mathrm{diff}}}/{d \psi _{0}}$",fontsize=20)
plt.xlabel(r"$\psi_{0}$",fontsize=20)

Text(0.5, 0, '$\\psi_{0}$')

In [40]:
#fig4
plt.figure(figsize=(4,4),dpi=300)
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

#计算psi_0=pi/4
psi_0 = torch.tensor(torch.pi/4, dtype=torch.float32) 
psi_flex = torch.arange(-torch.pi/2+0.01, torch.pi/2, 0.25)
psi_flex[-1] = torch.pi/2-0.01
psi_pitch = psi_pitch_from_psi_flex_and_psi_0(psi_flex,psi_0)
psi_roll = psi_roll_from_psi_flex_and_psi_0(psi_flex,psi_0)
psi_yaw = psi_yaw_from_psi_flex_and_psi_0(psi_flex,psi_0)
plt.plot(psi_flex,psi_pitch,label=r'$\psi_{\mathrm{pitch}},\psi_{0}=\frac{\pi}{4}$', linestyle='-',color='#FFBF00')
plt.plot(psi_flex,psi_roll,label=r'$\psi_{\mathrm{roll}},\psi_{0}=\frac{\pi}{4}$', linestyle='-',color='#16FF9A')
plt.plot(psi_flex,psi_yaw,label=r'$\psi_{\mathrm{yaw}},\psi_{0}=\frac{\pi}{4}$', linestyle='-',color='#C053FF')

#计算psi_0=-pi/4
psi_0 = torch.tensor(-torch.pi/4, dtype=torch.float32) 
psi_pitch = psi_pitch_from_psi_flex_and_psi_0(psi_flex,psi_0)
psi_roll = psi_roll_from_psi_flex_and_psi_0(psi_flex,psi_0)
psi_yaw = psi_yaw_from_psi_flex_and_psi_0(psi_flex,psi_0)
plt.plot(psi_flex,psi_pitch,label=r'$\psi_{\mathrm{pitch}},\psi_{0}=-\frac{\pi}{4}$', linestyle='--',linewidth=1,color='#FFBF00',marker='o',markersize='4')
plt.plot(psi_flex,psi_roll,label=r'$\psi_{\mathrm{roll}},\psi_{0}=-\frac{\pi}{4}$', linestyle='--',linewidth=1,color='#16FF9A',marker='o',markersize='4')
plt.plot(psi_flex,psi_yaw,label=r'$\psi_{\mathrm{yaw}},\psi_{0}=-\frac{\pi}{4}$', linestyle='--',linewidth=1,color='#C053FF',marker='o',markersize='4')

plt.grid()

x_ticks = np.arange(-np.pi/2, np.pi/2+0.0001, np.pi/6)
x_tick_labels = [r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{3}$', r'$-\frac{\pi}{6}$',
                 '0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$']
plt.xticks(x_ticks, x_tick_labels,fontsize=20)

y_ticks = np.arange(-np.pi/2, np.pi/2+0.0001, np.pi/6)
y_tick_labels = [r'$-\frac{\pi}{2}$',r'$-\frac{\pi}{3}$', r'$-\frac{\pi}{6}$',
                 '0', r'$\frac{\pi}{6}$', r'$\frac{\pi}{3}$',r'$\frac{\pi}{2}$']
plt.yticks(y_ticks, y_tick_labels,fontsize=20)
plt.axis('equal')
plt.legend(prop={'size':15},handlelength=1,borderpad=0.2, labelspacing=0.1)
plt.xlim(-np.pi/2,np.pi/2)
plt.ylim(-np.pi/2,np.pi/2)

(-1.5707963267948966, 1.5707963267948966)